### Embedding Model Evaluation
> Based on the use case / purpose, the embedding models can be evaluated with custom data  
> The custom data can represent the industry / domain. It can represent the use case (classification / clustering / similarity etc)  
> Depending on need, there can be multiple embedding model used within a system for different parts

#### Import of required Libraries

In [1]:
# Sentence transformers to use the embedding models locally
from sentence_transformers import SentenceTransformer, util
import pandas as pd

# Google AI library
from google import genai
from google.genai import types

# Load Environment variables from file
from dotenv import load_dotenv

# Initialise an client object with API key
load_dotenv ()
client = genai.Client()

# Import library for making the tests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Functions from SciPy for check / test
from scipy import spatial
from scipy.stats import pearsonr

import csv

c:\Users\imart\OneDrive\Documents\MyDev\Outskill\Repos\GenAIEngineering-Cohort3-My\Week5\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Utility Functions
> Functions defined for basic comparison and test  
> It can be re-used in different modules

In [2]:
def cosine_similarity(vec1, vec2):

    """
    Function to provide cosine similarity between given 2 vectors.
    Vectors are provided in the form of list of numbers
    Cosine Similarity is returned as a number. 1 being the highest representing high similarity
    """

    # spatial.distance.cosine returns the cosine distance
    # from the distance similarity can be computed, considering the unit vector
    
    return 1 - spatial.distance.cosine(vec1, vec2)

In [3]:
def get_cosine_similarities_HF(model, pairs):
    
    """
    Function to compute similarity scores in each pair for a given list of text pairs.
    Model from HF is given as an argument : as a sentence transformer model.
    For the Text pairs provided, embedding vectors are captured and consine similarities are provided as a list of numbers
    """

    similarities = []
    
    for s1, s2 in pairs:
                
        # Embed the texts
        emb1 = model.encode(s1, convert_to_tensor=True)
        emb2 = model.encode(s2, convert_to_tensor=True)
        
        # Identify the cosine similarity
        sim = util.cos_sim(emb1, emb2).item()
        similarities.append(sim)
    
    return similarities

In [4]:
def get_cosine_similarities_gemini(model, pairs):
    
    """
    Function to compute similarity scores in each pair for a given list of text pairs.
    Model from Gemini is given as an argument : Name of the embedding model as string.
    For the Text pairs provided, embedding vectors are captured and consine similarities are provided as a list of numbers    
    """

    similarities = []
    
    for s1, s2 in pairs:
        
        result = client.models.embed_content(
                model=model,        
                contents=[s1, s2],
                config=types.EmbedContentConfig(output_dimensionality=768)
                )

        sim = cosine_similarity (result.embeddings[0].values, result.embeddings[1].values)

        similarities.append(sim)

    return similarities

#### Make a Similarity Test
> Identify Known sentence pair, with known level of similarity  
> Calculate similarity score for each pair from various model  
> Compare the similarity scores from diff models for same text pair and compare

In [5]:
# Choose 2 embedders from HF to compare
a = "sentence-transformers/all-MiniLM-L6-v2"
b = "BAAI/bge-m3"
model_a = SentenceTransformer(a)
model_b = SentenceTransformer(b)

# Choose an embedding model from Gemini
g = "gemini-embedding-001"

In [6]:
# Set of known text with expected similarity.
sentence_pairs = [
    ("A man is eating pizza.", "A man is eating pizza."), # Same
    ("The weather was rainy last week.", "Currently I am wokring under hot sun"), # opposite
    ("AI helps techies", "Techies actuall help AI"), # opposite ..?
    ("He is playing flute.", "A person plays an instrument."),   # related
    ("I love dogs.", "She has a dog as pet."),      # somewhat related
    ("My car is red.", "My vehicle is painted Red"),        # similar
]

# Compute similarity list from the chosen HF models for same set of texts
sims_a = get_cosine_similarities_HF(model_a, sentence_pairs)
sims_b = get_cosine_similarities_HF(model_b, sentence_pairs)

# Compute similiarity list from the gemini model as well
sims_g = get_cosine_similarities_gemini (g, sentence_pairs)

# Result comparison
Result = pd.DataFrame({
    'Sentence 1': [p[0] for p in sentence_pairs],
    'Sentence 2': [p[1] for p in sentence_pairs],
    'Model A': sims_a,
    'Model B': sims_b,
    'Model G' : sims_g,
})

# Difference in similarity score between models
print ("Model A : "+a+"\nModel B : "+b+"\nModel G : "+g)

Result

Model A : sentence-transformers/all-MiniLM-L6-v2
Model B : BAAI/bge-m3
Model G : gemini-embedding-001


,Sentence 1,Sentence 2,Model A,Model B,Model G
0,A man is eating pizza.,A man is eating pizza.,1.000000,1.000000,1.000000
1,The weather was rainy last week.,Currently I am wokring under hot sun,0.170642,0.668367,0.598250
2,AI helps techies,Techies actuall help AI,0.907050,0.889847,0.891505
3,He is playing flute.,A person plays an instrument.,0.675566,0.843809,0.780592
4,I love dogs.,She has a dog as pet.,0.445857,0.765896,0.669298
5,My car is red.,My vehicle is painted Red,0.815809,0.927638,0.865017


**Similarity Test : Data Set**  
> Perform similarity test with pre defined data set, which is organised as text pairs and corresponding annotated score  
> Its a sample data from a pre-defined HF dataset  
> Similarly, a custom data set can be created, with annotated score Or enumerated range of score
> Finally its compared with correlation score

In [7]:
# Load the Test data from CSV file
Similarity_Sample = pd.read_csv ('Similarity_Data_sample.csv')

# Text pairs that are going to be used for similarity test
Test_Pairs = list (zip (Similarity_Sample['sentence1'], Similarity_Sample['sentence2']))

# Similarity score from Test Data as a reference
Test_Sim = Similarity_Sample['score'].to_list()

# Calculate pair wise similarity from 2 different models
sims_a = get_cosine_similarities_HF(model_a, Test_Pairs)
sims_b = get_cosine_similarities_HF(model_b, Test_Pairs)


In [8]:
sims_a

[0.9393033981323242,
 0.9020318388938904,
 0.8920009732246399,
 0.7945610880851746,
 0.9286527633666992,
 0.8919877409934998,
 0.33276817202568054,
 0.4605238437652588,
 0.6353746652603149,
 0.9613066911697388,
 0.7829393744468689,
 0.9642895460128784,
 0.8951709270477295,
 0.9853218197822571,
 0.5235110521316528,
 0.911227285861969,
 0.957695722579956,
 0.39119163155555725,
 0.933904767036438,
 0.9479299783706665,
 0.7157082557678223,
 0.5901291370391846,
 0.9502543210983276,
 0.7425967454910278,
 0.9549217224121094,
 0.9302327632904053,
 0.9767974615097046,
 0.8337551355361938,
 0.8997809886932373,
 0.8587252497673035,
 0.934901237487793,
 0.966414213180542,
 0.5438897013664246,
 0.6018918752670288,
 0.8823297619819641,
 0.7823505401611328,
 0.9802548289299011,
 0.9130768775939941,
 0.729139506816864,
 0.9564617276191711,
 0.8636921048164368,
 0.9150030612945557,
 0.342272013425827,
 0.7361303567886353,
 0.9795697331428528,
 0.2476482093334198,
 0.2744419276714325,
 0.712240397930145

In [9]:
sims_b

[0.9904777407646179,
 0.9488599896430969,
 0.9519258737564087,
 0.8851286768913269,
 0.9375605583190918,
 0.965636134147644,
 0.6924579739570618,
 0.8268412351608276,
 0.762356698513031,
 0.9833276271820068,
 0.9268436431884766,
 0.9844040870666504,
 0.9142899513244629,
 0.992712140083313,
 0.746533215045929,
 0.9406777620315552,
 0.9676914811134338,
 0.6989247798919678,
 0.956158459186554,
 0.9826555848121643,
 0.8372553586959839,
 0.7154467701911926,
 0.9854350686073303,
 0.9072666764259338,
 0.934386670589447,
 0.977813184261322,
 0.9839583039283752,
 0.8218039870262146,
 0.9557154178619385,
 0.930804967880249,
 0.9695488214492798,
 0.9811359643936157,
 0.7768548727035522,
 0.7538047432899475,
 0.8855119943618774,
 0.8958238959312439,
 0.9920382499694824,
 0.9585714936256409,
 0.7908986806869507,
 0.9414121508598328,
 0.9364256262779236,
 0.9452947378158569,
 0.7044517397880554,
 0.8493853807449341,
 0.9929148554801941,
 0.5983005166053772,
 0.554451048374176,
 0.8388973474502563,
 

**Pearson Correlation**  
It indicates how well the Test result (list of numbers) and the reference score (numbers) correlate  
Higher the number well correlated. It means Test result of Similarity score is aligned to what is present in Reference data   
Instead of absolute value check, the similarity scores can be enumerated based on range ( > 0.95 : Same; 0.8 .. 0.95 : Similar etc)

In [10]:
# Compute Pearson correlation : Model A
corr_a, p_value_a = pearsonr(Test_Sim, sims_a)
print(f"Model A : Pearson correlation: {corr_a:.4f}")

# Compute Pearson correlation : Model B
corr_b, p_value_b = pearsonr(Test_Sim, sims_b)
print(f"Model B : Pearson correlation: {corr_b:.4f}")

Model A : Pearson correlation: 0.9301
Model B : Pearson correlation: 0.8944


#### Make a Classification Test
> When text are vectorised by embedding model, the resulting vectors (number array) can be treated as classification features  
> If there is a pre-defined classification labels for the text, the corrsponding vectors from the model can be used for creating a classification model and a test can be made
> This will indicate how well the vectors are grouped cohesively in the vector space for the given set of classes

In [11]:
# Define a Test data set with text and corresponding class label
# this label can be representing various tags - depending on the use case
texts = [
        "I love this movie!",
        "This was terrible.",
        "Amazing performance.",
        "Not worth watching.",
        "It was okay, not great.",
        "Absolutely fantastic!"
]
labels = ['Positive', 'Negative','Positive', 'Negative','Negative','Positive']

# Split the data set for training and testing
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.3, random_state=42)


In [12]:
X_train #Input text data used for training the model
X_test #Input text data used for testing (evaluation)
y_test #Class labels (correct answers) for the test set
y_train #Class labels (correct answers) for the training set

['Positive', 'Positive', 'Negative', 'Negative']

> For Each model, make the embddings for both training data and test data  
> Make a simple classification model, based on vector data and corresponding label for the Test data split  
> Once the model is built, for the Test vectors predict the class  
> make comparison for accuracy of class predicted vs the class label from the test data set

In [13]:
# The models for comparison
models = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2"
]

for name in models:
    model = SentenceTransformer(name)
    train_emb = model.encode(X_train)
    test_emb = model.encode(X_test)

    cl_model = LogisticRegression(max_iter=1000)
    cl_model.fit(train_emb, y_train)
    preds = cl_model.predict(test_emb)

    acc = accuracy_score(y_test, preds)
    print(f"{name} → Accuracy: {acc:.3f}")

sentence-transformers/all-MiniLM-L6-v2 → Accuracy: 1.000
sentence-transformers/all-mpnet-base-v2 → Accuracy: 1.000


**Data Set : Industry operation**  
Pick a data set that represents a industry data.  
Each text provided mentions a process step / operation / requirement from an industry  
the classification label provides the department in the organisation which holds the responsibility / relevant to the process  
Similarly vectorise the texts, train and test the classification model  
The score can indicate how well vectors from 2 models can captured the nuance of the industry / organisation, when used in **classification**

In [14]:
# Read Sample data from CSV
Data = pd.read_csv ('Process_n_Departments.csv')

# Consider the text and label form the data frame
texts = Data['Process'].to_list ()
labels = Data['Department'].to_list ()

# Split
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.4, random_state=42)


**Visualise**  
> The vectors that are made from embedding model and the corresponding label can be used for visualisation  
> High dimensionality vector cannot be visualised, unless it is reduced in dimension  
> Store them as tab separated file and can be loaded in https://projector.tensorflow.org/

In [15]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
Vectors = model.encode (texts, convert_to_numpy=True)

# Write to tab-separated file

with open('vectors.csv', "w", newline="", encoding="utf-8") as f:
    
    writer = csv.writer(f, delimiter='\t')
    
    for V in Vectors:
        writer.writerow(V.tolist())

with open('labels.csv', "w", newline="", encoding="utf-8") as f:
    
    writer = csv.writer(f, delimiter='\t')
    
    for L in labels:
        writer.writerow([L])

You’re absolutely creating a classifier model — but you’re doing it using a hybrid ML approach that mixes deep learning embeddings with a traditional ML classifier.

Let’s break it down step-by-step 👇

🧩 Step 1: What is happening conceptually

When you run this code:

for name in models:
    model = SentenceTransformer(name)
    train_emb = model.encode(X_train)
    test_emb = model.encode(X_test)

    cl_model = LogisticRegression(max_iter=1000)
    cl_model.fit(train_emb, y_train)
    preds = cl_model.predict(test_emb)

    acc = accuracy_score(y_test, preds)
    print(f"{name} → Accuracy: {acc:.3f}")


You’re doing two major things:

🧠 Part 1: Text → Embeddings (Feature Extraction)
model = SentenceTransformer(name)
train_emb = model.encode(X_train)
test_emb = model.encode(X_test)


This step uses a pre-trained deep learning model (like all-MiniLM-L6-v2 or all-mpnet-base-v2)
to turn your text sentences into numerical embeddings (vectors).

This model already understands language very well (trained on millions of sentences).

It’s not learning here — it’s just converting text into a format that ML models can understand.

💡 So here you are not training the transformer, you are just using it as a feature extractor.

⚙️ Part 2: Embeddings → Classification
cl_model = LogisticRegression(max_iter=1000)
cl_model.fit(train_emb, y_train)


Here, you train your own classifier model (a machine learning model).

You give it the embeddings (train_emb) and the correct labels (y_train).

The classifier learns patterns like:

Embeddings that look like this → “Finance”

Embeddings that look like that → “Admin”

etc.

Then you test it:

preds = cl_model.predict(test_emb)


That’s your custom trained classifier making predictions on unseen text.

⚖️ Step 2: So what exactly are you building?

✅ You are creating a classifier model, but not from scratch.

Instead, you’re using:

A pre-trained deep learning model (SentenceTransformer) → to get text embeddings (language understanding part)

A machine learning model (Logistic Regression) → to learn how to map those embeddings to your specific classes (classification part)

So the overall system is indeed a text classifier — for your use case (like department detection or sentiment analysis).

🧱 Step 3: The two-layer architecture (simple visualization)
                ┌──────────────────────────────┐
                │ Pre-trained Transformer Model │
                │ (e.g., all-MiniLM-L6-v2)     │
                └─────────────┬────────────────┘
                              │
               "text input" → │ → Embedding Vector (numbers)
                              │
                ┌─────────────▼──────────────┐
                │ Logistic Regression Model  │
                │ (your classifier)          │
                └─────────────┬──────────────┘
                              │
                              ▼
                    Predicted Label (e.g., "Finance")

🧠 Step 4: Why this is a powerful approach

You get the best of both worlds:

Component	Job	Who does it
Text understanding	Extracts meaning from words	SentenceTransformer (pre-trained)
Classification	Learns your custom categories	Logistic Regression (trained by you)

This approach:

Works even with small datasets

Avoids training large deep models from scratch

Is fast and interpretable

Gives strong accuracy for text classification

🧪 Step 5: After training

After this loop finishes, you have trained classifiers for each embedding model.

Example:

cl_model = LogisticRegression(max_iter=1000)
cl_model.fit(train_emb, y_train)


That cl_model object is your classifier.
You can save it and use it later to predict new text:

new_text = ["Approve vendor payment invoices"]
new_emb = model.encode(new_text)
prediction = cl_model.predict(new_emb)
print(prediction)
# Output: ['Finance']


Now your classifier model can classify new unseen text 💪

✅ Summary
Concept	Meaning
SentenceTransformer	Pre-trained deep learning model used to convert text → embeddings
Logistic Regression	Machine learning model trained by you to classify those embeddings
Your result	A full text classification pipeline
Type of model	A classifier model (but hybrid — DL embeddings + ML classifier)

So yes ✅,
➡️ You are creating a classifier model
➡️ that uses Sentence Transformers for text understanding
➡️ and Logistic Regression for actual classification.